In [ ]:
import caiman as cm
import tifffile
import os
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from cell_registration import *
import holoviews as hv
hv.extension('bokeh')
from tkinter import filedialog

Set paths

In [ ]:
astro3 = CellReg()
N = 5 #number of sessions
scale_factor = 2
print('Animal: '+astro3.animal, ' FOV: '+astro3.FOV)
print('Base directory: ' + astro3.base_directory)
print('Metadata file: ' + astro3.metadata_file)

#### load data

first, select max projection image files

In [ ]:
max_im_files=astro3.get_summary_images(image_type='max dff')
max_images = [tifffile.imread(f) for f in max_im_files]


In [ ]:
max_im_files

select footprint files (not registered - for plotting over images) & convert footprints to masks\
---must convert to masks if imported footprints from inscopix helper files.

In [2]:
footprints=astro3.load_footprints_3D()


NameError: name 'astro3' is not defined

### select registered footprint files (do not plot over images)

In [6]:
reg_foots = astro3.load_shifted_footprints_2D()

#### plot overlay of desired sessions aligned footprints

In [7]:
astro3.plot_overlay_footprints(scale_factor=scale_factor)

:Image   [x,y]   (z)

#### plot one session rois

In [8]:
astro3.rois_plot(session_ind=0,image=max_images[0])

NameError: name 'max_images' is not defined

#### plot all sessions all cells

In [11]:
plot_dict={i:astro3.rois_plot(session_ind=i,image=max_images[i],scale_factor=scale_factor,max_pct=99) for i in range(N)}
hv.HoloMap(plot_dict)

:HoloMap   [Default]
   :Overlay
      .Image.I  :Image   [x,y]   (z)
      .Image.II :Image   [x,y]   (z)

evalaute cell reg output for fc & another session

In [ ]:
reg_ind = get_reg_ind(ani,fov,file_key,base_dir)
#print(reg_ind)
#print(reg_ind[0][sID])  


In [ ]:
#name session pair for comparison with FC according to session number (i.e. session 1,2,3,4 = GEN1/Recall,GEN2/EXT1, etc) 
sID = 2
#generate an array of two lists containing cell IDs for putatively registered cells across compared sessions
session_comparison_array = [reg_ind[:,0][(reg_ind[:,0]>=0)&(reg_ind[:,sID]>=0)],reg_ind[:,sID][(reg_ind[:,0]>=0)&(reg_ind[:,sID]>=0)]]
print(session_comparison_array)


In [ ]:
np.unique(foots[0])


only registered cells

In [ ]:
#Does not work 
#reg_dict={i: rois_plot(foots[i],max_images[i],idxs=session_comparison_array[i],max_pct=99.5) for i in range(2)}
#reg_foots = {i: hv.Image(foots[i][session_comparison_array[i],:,:].sum(axis=0)).opts(width=foots[i].shape[2]*scale_factor,height=foots[i].shape[1]*scale_factor) for i in range(2)}
#(hv.HoloMap(reg_dict)+hv.HoloMap(reg_foots)).cols(2)

In [ ]:
#Plot only registered cells
reg_dict={i: rois_plot(foots[0],max_images[0],idxs=session_comparison_array[0],max_pct=99.5)+\
          rois_plot(foots[sID],max_images[sID],idxs=session_comparison_array[1],max_pct=99.5)}
reg_foots = {i: hv.Image(foots[0][session_comparison_array[0],:,:].sum(axis=0)).opts(width=foots[0].shape[2]*scale_factor,height=foots[0].shape[1]*scale_factor)+\
             hv.Image(foots[sID][session_comparison_array[1],:,:].sum(axis=0)).opts(width=foots[sID].shape[2]*scale_factor,height=foots[sID].shape[1]*scale_factor)}
(hv.HoloMap(reg_dict)+hv.HoloMap(reg_foots))

In [ ]:
np.unique(foots[0])

In [ ]:
#Same as above but allows for scrolling through each session pair
roi_list = [[session_comparison_array[0][i],session_comparison_array[1][i]] for i in range(len(session_comparison_array[0]))]
regdict_singles = {i: roi_plot(foots[0],roi_list[i][0],max_images[0],max_pct=99,scale_factor=scale_factor)+\
                  roi_plot(foots[sID],roi_list[i][1],max_images[sID],max_pct=99,scale_factor=scale_factor) for i in range(len(session_comparison_array[0]))}
hv.HoloMap(regdict_singles).collate()



In [ ]:
#Input list of bad cell pairs as bad_cell_reg_pairs
bad_cell_reg_pairs = [0,1,2,3,4,5,6,8,10,11,12,14,15,16,17,19,20,21,22,23,24]
print(reg_ind)
#Remove bad cell registration pairs from session_comparison_array and create new numpy.ndarray CorrSCA with updated values
CorrSCA = (np.delete(session_comparison_array[0], bad_cell_reg_pairs), np.delete(session_comparison_array[1], bad_cell_reg_pairs))
for i in session_comparison_array[0]:
    if i not in CorrSCA[0]:
        reg_ind[i][sID] = -1
#print(session_comparison_array)
print(len(CorrSCA[0]))
print(reg_ind)
#print(len(bad_cell_reg_pairs))
#Reg_Across_Days = []
#print(session_comparison_array[0])

In [ ]:
Overlap_AD = [4,0,0,0]

In [ ]:
np.savetxt("astro_10_cell_reg.csv", reg_ind, delimiter=",")

In [ ]:
np.unique(foots[0])

debug

In [ ]:
roi_sets = [foots[0].sum(axis=0),foots[0][fc_recall[0],:,:].sum(axis=0)]

In [ ]:
np.unique(roi_sets[0])

In [ ]:
hv.Image(roi_sets[0])

In [ ]:
hv.Image(roi_sets[0]+roi_sets[1])

In [ ]:
rois_plot(foots[0],image=None,cmap='gray')

In [ ]:
rois_plot(foots[0],image=None)